# **OSEMN | Obtain: Data Extraction | CDS Link**

CDS files were programmatically downloaded using Python, with additional handling for webpage links and logging of extraction outcomes. The PDF collection will be used for data sxtraction, validation and integration with IPEDS data.

**1. Environment Setup: Mount Drive to upload in Google Drive**

The input file is uploaded in Google Drive. Google Drive is mounted to acces the files and save the extracted files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**2. Loading the CDS Link Dataset**

The CDS link dataset is uploaded in Google Colab using the pandas library. This dataset contains institution names and their corresponding CDS URLs, which serve as the main input for the PDF download process. Basic checks are performed after loading the file to confirm that the dataset has been read correctly and that the required columns are available for extraction.

In [ ]:
import pandas as pd

input_file = "/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/02_data_scraping_out/v2_cds_links_60_institutions.xlsx" # Define the path to the input Excel file
df = pd.read_excel(input_file) # read the Excel file into a pandas DataFrame

print(df.head())
print(df.columns.tolist())
print(df.shape)

   #           Institution                                            CDS URL  \
0  1    Harvard University            https://oir.harvard.edu/common-data-set   
1  2       Yale University               https://oir.yale.edu/common-data-set   
2  3  Princeton University          https://oir.princeton.edu/common-data-set   
3  4   Columbia University           https://irp.columbia.edu/common-data-set   
4  5      Brown University  https://oir.brown.edu/institutional-data/commo...   

                Notes  
0  Full CDS available  
1  Full CDS available  
2  Full CDS available  
3  Full CDS available  
4  Full CDS available  
['#', 'Institution', 'CDS URL', 'Notes']
(60, 4)


**3. Check the Important Columns**

This is to inspect the structure of the dataset and confirm that the required fields are present. In particular, the institution name and CDS link columns are checked because these are the main inputs for the PDF extraction process. This helps avoid errors caused by incorrect column names or missing data.

In [ ]:
print(df.columns.tolist())

['#', 'Institution', 'CDS URL', 'Notes']


**4. Keep only rows with valid CDS links**

This separates valid CDS links from missing entries. The number of valid CDS links is compared against the total number of institutions to assess the coverage of the CDS acquisition process.


In [ ]:
#standardise the 'CDS URL' column by converting values to strings and removing unnecessary whitespace.
df["CDS URL"] = df["CDS URL"].astype(str).str.strip()

#filter teh dataset to retain only rows containing valid CDS URLs
valid_df = df[
    df["CDS URL"].notna() &
    (df["CDS URL"] != "") &
    (df["CDS URL"].str.lower() != "nan")
].copy()

print("Total rows:", len(df))
print("Valid CDS links:", len(valid_df))

valid_df.head()

Total rows: 60
Valid CDS links: 60


,#,Institution,CDS URL,Notes
0,1,Harvard University,https://oir.harvard.edu/common-data-set,Full CDS available
1,2,Yale University,https://oir.yale.edu/common-data-set,Full CDS available
2,3,Princeton University,https://oir.princeton.edu/common-data-set,Full CDS available
3,4,Columbia University,https://irp.columbia.edu/common-data-set,Full CDS available
4,5,Brown University,https://oir.brown.edu/institutional-data/commo...,Full CDS available


**5. Download CDS PDFs:**
1. creates a folder
2. downloads each file
3. records success or failure

In [ ]:
import os
import requests
import re
import time
from urllib.parse import urlparse

download_folder = "/content/drive/MyDrive/P2 MDS/01_Data_Acquisition/03_final_extraction/CDS_data"
os.makedirs(download_folder, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}
download_results = []

#download the CDS file
for idx, row in valid_df.iterrows():
    institution = str(row["Institution"]).strip()
    url = str(row["CDS URL"]).strip()

    print(f"Downloading {institution} ...")

    #create filename
    safe_name = re.sub(r'[^A-Za-z0-9]+', '_', institution).strip('_')
    parsed = urlparse(url)
    original_name = os.path.basename(parsed.path)

    if original_name.lower().endswith(".pdf"):
        file_name = f"{safe_name}.pdf"
    else:
        file_name = f"{safe_name}.pdf"

    file_path = os.path.join(download_folder, file_name)

    try:
        #request for longer timeout
        r = requests.get(url, headers=headers, timeout=90)
        r.raise_for_status()

        content_type = r.headers.get("Content-Type", "").lower()

        #save only if content looks like pdf
        if "pdf" in content_type or url.lower().endswith(".pdf"):
            with open(file_path, "wb") as f:
                f.write(r.content)

            download_results.append({
                "Institution": institution,
                "CDS Link": url,
                "Saved File": file_name,
                "Status": "Downloaded"
            })

        else:
            download_results.append({
                "Institution": institution,
                "CDS Link": url,
                "Saved File": None,
                "Status": f"Not a direct PDF ({content_type})"
            })

    except Exception as e:
        download_results.append({
            "Institution": institution,
            "CDS Link": url,
            "Saved File": None,
            "Status": f"Failed: {e}"
        })

    #pause for 2 secs before the next request
    time.sleep(2)

download_log = pd.DataFrame(download_results)
download_log.head()

,Institution,CDS Link,Saved File,Status
0,Harvard University,https://oir.harvard.edu/common-data-set,None,Failed: 504 Server Error: Gateway Time-out for...
1,Yale University,https://oir.yale.edu/common-data-set,None,Not a direct PDF (text/html; charset=utf-8)
2,Princeton University,https://oir.princeton.edu/common-data-set,None,Failed: HTTPSConnectionPool(host='oir.princeto...
3,Columbia University,https://irp.columbia.edu/common-data-set,None,Failed: HTTPSConnectionPool(host='irp.columbia...
4,Brown University,https://oir.brown.edu/institutional-data/commo...,None,Not a direct PDF (text/html; charset=utf-8)


**6. Notes**

Some CDS links may not point directly to PDF files. Instead, they lead to institutional webpages, directories or research office pages where the CDS file is listed separately. These cases are recorded in the download log rather than discarded. This allows them to be reviewed manually and, where possible, resolved in a later extraction stage.